# VADER and BERT baselines, and language identification
Input: `marakech.csv` (collected reviews, not redistributed) merged with the Gemini outputs (`llm_*` columns).
Output: `baselines_predictions.csv`; the released file `data/baselines_vader_bert.csv` contains its prediction columns only.

Preprocessing (identical for the three methods): the review text is trimmed of leading/trailing whitespace; the main analysis uses its first 500 characters, the sensitivity analysis the full text. No other cleaning is applied and no review is removed.

In [ ]:
!pip install -q vaderSentiment transformers torch scikit-learn pandas numpy scipy langdetect

In [ ]:
import re
import numpy as np
import pandas as pd

TRUNC_CHARS = 500          # same truncation as the Gemini run
TEXT_COL = 'text'
df = pd.read_csv('LLM_FULL_ANALYSIS.csv')   # collected reviews + Gemini outputs
print(len(df), 'reviews')

In [ ]:
# Star rating -> star-derived label (4-5 Positive, 3 Neutral, 1-2 Negative)
def parse_rating(v):
    if pd.isna(v): return np.nan
    m = re.search(r'(\d+(?:\.\d+)?)', str(v)); return float(m.group(1)) if m else np.nan
def stars_to_label(s):
    if pd.isna(s): return np.nan
    return 'Positive' if s >= 4 else ('Neutral' if s == 3 else 'Negative')
df['stars'] = df['rating'].apply(parse_rating)
df['gt'] = df['stars'].apply(stars_to_label)
n0 = len(df); df = df[df['gt'].notna() & df[TEXT_COL].notna()].reset_index(drop=True)
print('rows removed:', n0 - len(df))                      # 0 in this corpus
df['input_full'] = df[TEXT_COL].astype(str).str.strip()
df['input_trunc'] = df['input_full'].str[:TRUNC_CHARS]
print('reviews >', TRUNC_CHARS, 'characters:', (df['input_full'].str.len() > TRUNC_CHARS).sum())

In [ ]:
# Language identification (Section 3.2): langdetect with a fixed seed
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0
def lang(t):
    try: return detect(t)
    except Exception: return 'unknown'
df['lang'] = df['input_full'].apply(lang)
print('English:', (df['lang'] == 'en').sum(), '/', len(df), f"({(df['lang'] == 'en').mean()*100:.2f}%)")   # 14,835 / 14,838

In [ ]:
# VADER (compound >= 0.05 Positive, <= -0.05 Negative, otherwise Neutral)
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
vader = SentimentIntensityAnalyzer()
def vader_label(text):
    c = vader.polarity_scores(str(text))['compound']
    return 'Positive' if c >= 0.05 else ('Negative' if c <= -0.05 else 'Neutral')
df['vader_full'] = df['input_full'].apply(vader_label)
df['vader_trunc'] = df['input_trunc'].apply(vader_label)

In [ ]:
# BERT: nlptown/bert-base-multilingual-uncased-sentiment (1-5 stars -> 3 classes); a GPU is recommended
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
MODEL = 'nlptown/bert-base-multilingual-uncased-sentiment'
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device).eval()

@torch.no_grad()
def bert_stars(texts, batch_size=64):
    out = []
    for i in range(0, len(texts), batch_size):
        enc = tok([str(t) for t in texts[i:i + batch_size]], truncation=True, max_length=512,
                  padding=True, return_tensors='pt').to(device)
        out.extend((model(**enc).logits.argmax(-1) + 1).cpu().tolist())
    return out

df['bert_stars_full'] = bert_stars(df['input_full'].tolist())
df['bert_stars_trunc'] = bert_stars(df['input_trunc'].tolist())
df['bert_full'] = df['bert_stars_full'].apply(stars_to_label)
df['bert_trunc'] = df['bert_stars_trunc'].apply(stars_to_label)

In [ ]:
# LLM polarity: the 16 off-schema "Mixed" outputs are grouped with Neutral (Figure 3)
VALID = {'Positive', 'Neutral', 'Negative'}
raw = df['llm_sentiment'].astype(str).str.strip()
df['llm_pred'] = raw.where(raw.isin(VALID), 'Neutral')
df.to_csv('baselines_predictions.csv', index=False)
print('saved baselines_predictions.csv  (all tables are produced by reproduce_all.ipynb)')